# Dynamic / Interactive Visualizations – Plotly

Run in Colab or Jupyter — charts are fully interactive.

In [ ]:
# # !pip install plotly -q

# import pandas as pd
# import numpy as np
# import plotly.express as px
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import os, warnings
# warnings.filterwarnings('ignore')

# try:
#     df = pd.read_csv('data/demand_clean.csv', parse_dates=['Date'])
# except FileNotFoundError:
#     from google.colab import files
#     print("Upload demand_clean.csv")
#     uploaded = files.upload()
#     import io
#     df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]), parse_dates=['Date'])

# df['year']    = df['Date'].dt.year
# df['month']   = df['Date'].dt.month
# df['quarter'] = df['Date'].dt.quarter
# print("Ready:", df.shape)

In [1]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os, warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('data/demand_clean.csv', parse_dates=['Date'])
df['year']    = df['Date'].dt.year
df['month']   = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter
print("Ready:", df.shape)

Ready: (1048575, 8)


## 1. Interactive Monthly Trend

In [3]:
monthly = (df.dropna(subset=['Date'])
             .groupby(df['Date'].dt.to_period('M'))['Order_Demand']
             .sum().reset_index())
monthly.columns = ['period','demand']
monthly['period'] = monthly['period'].dt.to_timestamp()

fig = px.line(monthly, x='period', y='demand',
              title='Monthly Demand Trend',
              labels={'period':'Month','demand':'Order Demand'})
fig.update_traces(line_color='steelblue', line_width=2)
fig.update_layout(hovermode='x unified')
fig.show()

## 2. Demand by Warehouse – Interactive Bar

In [4]:
by_wh = df.groupby('Warehouse')['Order_Demand'].sum().reset_index()
by_wh.columns = ['Warehouse','Total_Demand']

fig = px.bar(by_wh.sort_values('Total_Demand', ascending=True),
             x='Total_Demand', y='Warehouse',
             orientation='h',
             title='Total Demand by Warehouse',
             color='Total_Demand',
             color_continuous_scale='Blues',
             text='Total_Demand')
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(coloraxis_showscale=False, showlegend=False)
fig.show()

## 3. Category Breakdown – Pie / Sunburst

In [5]:
by_cat = df.groupby('Product_Category')['Order_Demand'].sum().reset_index()

fig = px.pie(by_cat, values='Order_Demand', names='Product_Category',
             title='Demand Share by Product Category',
             hole=0.35)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 4. Yearly Trend – Animated Bar Race

In [6]:
yr_wh = df.groupby(['year','Warehouse'])['Order_Demand'].sum().reset_index()

fig = px.bar(yr_wh, x='Warehouse', y='Order_Demand', color='Warehouse',
             animation_frame='year',
             title='Demand per Warehouse Over the Years',
             range_y=[0, yr_wh['Order_Demand'].max() * 1.1])
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.show()

## 5. Heatmap – Interactive

In [7]:
hm = df.groupby(['Warehouse','month'])['Order_Demand'].sum().reset_index()
hm.columns = ['Warehouse','Month','Demand']

fig = px.density_heatmap(hm, x='Month', y='Warehouse', z='Demand',
                         color_continuous_scale='YlOrRd',
                         title='Demand Heatmap – Warehouse × Month',
                         text_auto=True)
fig.show()

## 6. Top 20 Products – Interactive Pareto

In [8]:
top20 = df.groupby('Product_Code')['Order_Demand'].sum().nlargest(20).reset_index()
top20['cumulative_pct'] = top20['Order_Demand'].cumsum() / top20['Order_Demand'].sum() * 100

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=top20['Product_Code'], y=top20['Order_Demand'], name='Demand',
           marker_color='cornflowerblue'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=top20['Product_Code'], y=top20['cumulative_pct'],
               name='Cumulative %', mode='lines+markers', line=dict(color='tomato')),
    secondary_y=True
)
fig.add_hline(y=80, line_dash='dash', line_color='gray', secondary_y=True)
fig.update_layout(title='Top 20 Products – Pareto', hovermode='x unified')
fig.show()

## 7. Scatter – Orders vs Demand (by Category)

In [9]:
scatter_df = (df.groupby(['Product_Code','Product_Category'])
                .agg(total_demand=('Order_Demand','sum'),
                     order_count=('Order_Demand','count'))
                .reset_index())

fig = px.scatter(scatter_df, x='order_count', y='total_demand',
                 color='Product_Category',
                 hover_data=['Product_Code'],
                 title='Order Count vs Total Demand',
                 labels={'order_count':'# Orders','total_demand':'Total Demand'},
                 opacity=0.7, size_max=12,
                 log_x=True, log_y=True)
fig.show()